In [1]:
# 1. 导入依赖库（与训练代码一致）
# ======================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from rdkit import Chem
from mordred import Calculator, descriptors
import joblib
import warnings
warnings.filterwarnings("ignore")

# 全局Mordred计算器
calc = Calculator(descriptors, ignore_3D=True)

# ======================================

In [2]:
# 2. 复用训练时的核心函数/类
# ======================================
def get_toxicity_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(1613, dtype=np.float32)
    desc = calc(mol)
    desc = desc.fill_missing(0)
    desc_array = np.array(desc, dtype=np.float32)
    desc_array[np.isinf(desc_array)] = 0
    return desc_array

class ReproductiveToxicityMLP(nn.Module):
    def __init__(self, compound_dim, n_species=10, species_emb_dim=16, hidden_dims=[128, 64]):
        super().__init__()
        self.species_emb = nn.Embedding(num_embeddings=n_species, embedding_dim=species_emb_dim)
        input_dim = compound_dim + species_emb_dim
        
        layers = []
        prev_dim = input_dim
        for dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, dim))
            layers.append(nn.LeakyReLU(0.1))
            layers.append(nn.Dropout(0.2))
            prev_dim = dim
        
        layers.append(nn.Linear(prev_dim, 1))
        layers.append(nn.Sigmoid())
        self.mlp = nn.Sequential(*layers)

    def forward(self, compound_feat, species_id):
        species_emb = self.species_emb(species_id)
        x = torch.cat([compound_feat, species_emb], dim=1)
        return self.mlp(x)

# 单条预测函数
def predict_toxicity(model, smiles, species, species2id, scaler, device="cpu"):
    model.to(device)
    model.eval()
    
    if species not in species2id:
        species_id = 0
    else:
        species_id = species2id[species]
    
    feat = get_toxicity_descriptors(smiles)
    feat = scaler.transform([feat])[0]
    
    feat = torch.tensor(feat, dtype=torch.float32).unsqueeze(0).to(device)
    species_id = torch.tensor([species_id], dtype=torch.long).to(device)
    
    with torch.no_grad():
        prob = model(feat, species_id).cpu().numpy()[0][0]
    return float(max(0, min(1, prob)))

# ======================================

In [3]:
# 3. 多模型批量预测主函数
# ======================================
def predict_all_models():
    # ===================== 固定配置 =====================
    # 模型列表：(模型名, 权重路径, 物种数, 物种字典)
    MODEL_LIST = [
        # 格式：(模型名, 权重路径, 物种数, 物种字典)
        ("modelKE1076", "model2_KE1076.pth", 5, {'Chicken': 0, 'Homo sapiens': 1, 'Mouse ': 2, 'Rat': 3, 'Vizcacha': 4}),
        ("modelKE2251", "model2_KE2251.pth", 1, {'Homo sapiens': 0}),
        ("modelKE2294", "model2_KE2251.pth", 1, {'Homo sapiens': 0}),
        ("modelKE531", "model2_KE531.pth", 6, {'Cows': 0, 'Homo sapiens': 1, 'Macaca fascicularis': 2, 'Rabbit': 3, 'Rat': 4, 'Sheep': 5}),
        ("modelMIE2155", "model2_MIE2155.pth", 8, {'Cavia porcellus': 0, 'Cows': 1, 'Homo sapiens': 2, 'Lepidorhombus boscii': 3, 'Mouse': 4, 'Mullus barbatus': 5, 'Rat': 6, 'Sheep': 7}),
        ("modelMIE2293", "model2_MIE2293.pth", 19, {'Bacillus licheniformis': 0, 'Bacillus subtilis': 1, 'Cyprinus carpio': 2, 'Drosophila melanogaster': 3, 'Echinochloa phyllopogon': 4, 'Helicoverpa zea': 5, 'Homo sapiens': 6, 'Hypophthalmichthys molitrix': 7, 'Lolium rigidum': 8, 'Nicotiana tabacum': 9, 'Phanerodontia chrysosporium': 10, 'Phelipanche ramosa': 11, 'Priestia megaterium': 12, 'Rabbit': 13, 'Rat': 14, 'Rhodococcus sp. (in: high G+C Gram-positive bacteria)': 15, 'Silurus glanis': 16, 'Solanum tuberosum': 17, 'synthetic construct': 18}),
        ("modelKE530", "model2_KE530.pth", 1, {'Homo sapiens': 0}),
        ("modelKE1695", "model2_KE1695.pth", 4, {'Homo sapiens': 0, 'Mouse': 1, 'Rabbit': 2, 'Rat': 3}),
        ("modelKE2303", "model2_KE2303.pth", 5, {'Homo sapiens': 0, 'Mouse': 1, 'Rabbit': 2, 'Rat': 3, 'Xenopus laevis': 4}),
        ("modelKE2137", "model2_KE2137.pth", 7, {'African catfish': 0, 'Cows': 1, 'Gilthead sea bream': 2, 'Homo sapiens': 3, 'Monkey': 4, 'Rat': 5, 'Sheep': 6}),
        ("modelMIE1065", "model2_MIE1065.pth", 2, {'Homo sapiens': 0, 'Mouse': 1}),   
        ("modelKE1985", "model2_KE1985.pth", 1, {'Homo sapiens': 0}),    
        ("modelKE1047", "model2_KE1047.pth", 1, {'Homo sapiens': 0}),
        ("modelKE2402", "model2_KE2402.pth", 11, {'Brown trout': 0, 'Cat': 1, 'Killifish': 2, 'Labeo catla': 3, 'Mouse': 4, 'Pig': 5, 'Rabbit': 6, 'Rainbow trout': 7, 'Rat': 8, 'Zebrafish': 9, 'follicles': 10}),    
        ("modelKE1075", "model2_KE1075.pth", 5, {'Cows': 0, 'Horses': 1, 'Mouse': 2, 'Pig': 3, 'Rat': 4}),
        ("modelKE1050", "model2_KE1050.pth", 3, {'Homo sapiens': 0, 'Mouse': 1, 'Rat': 2}),    
        ("modelKE1972", "model2_KE1972.pth", 5, {'Chicken': 0, 'Female fathead minnows': 1, 'Homo sapiens': 2, 'Mouse': 3, 'Rat': 4}),
        ("modelKE1973", "model2_KE1973.pth", 11, {'Chicken': 0, 'Chironomus riparius': 1, 'Earthworm': 2, 'Fish': 3, 'Homo sapiens': 4, 'Laodelphax striatellus': 5, 'Mouse': 6, 'Mussel': 7, 'Pig': 8, 'Rat': 9, 'Zebrafish': 10}),    
        ("modelMIE1046", "model2_MIE1046.pth", 1, {'Homo sapiens': 0}),
        ("modelMIE2250", "model2_MIE2250.pth", 1, {'Homo sapiens': 0}),    
    ]
    # scaler路径
    SCALER_DICT = {
        "modelKE1076": "scaler_KE1076.pkl",
        "modelKE2251": "scaler_KE2251.pkl",
        "modelKE2294": "scaler_KE2294.pkl",
        "modelKE531": "scaler_KE531.pkl",
        "modelMIE2155": "scaler_MIE2155.pkl",
        "modelMIE2293": "scaler_MIE2293.pkl",
        "modelKE530": "scaler_KE530.pkl",
        "modelKE1695": "scaler_KE1695.pkl",
        "modelKE2303": "scaler_KE2303.pkl",
        "modelKE2137": "scaler_KE2137.pkl",
        "modelMIE1065": "scaler_MIE1065.pkl",
        "modelKE1985": "scaler_KE1985.pkl",
        "modelKE1047": "scaler_KE1047.pkl",
        "modelKE2402": "scaler_KE2402.pkl",
        "modelKE1075": "scaler_KE1075.pkl",
        "modelKE1050": "scaler_KE1050.pkl",
        "modelKE1972": "scaler_KE1972.pkl",
        "modelKE1973": "scaler_KE1973.pkl",
        "modelMIE1046": "scaler_MIE1046.pkl",
        "modelMIE2250": "scaler_MIE2250.pkl"   
    }    
    # ===================== 配置 =====================
    COMPOUND_DIM = 1613
    INPUT_EXCEL = "chemicaltopredict.xlsx"
    OUTPUT_EXCEL = "KE_MIE_毒性概率结果.xlsx"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"使用设备: {device}\n")

    # 读取数据
    df = pd.read_excel(INPUT_EXCEL)
    
    # ✅ 自动生成 chemical ID（从1开始编号）
    df.insert(0, "chemical ID", range(1, len(df)+1))
    
    smiles_list = df["smiles"].tolist()
    species_list = df["species"].tolist()

    # 逐个模型预测
    for model_name, weight_path, n_species, species2id in MODEL_LIST:
        print(f"===== 预测: {model_name} =====")
        col_name = model_name.replace("model", "")
        
        # 加载模型
        model = ReproductiveToxicityMLP(compound_dim=COMPOUND_DIM, n_species=n_species)
        model.load_state_dict(torch.load(weight_path, map_location=device, weights_only=True))
        model.eval()
        
        # 加载scaler
        scaler = joblib.load(SCALER_DICT[model_name])
        
        # 预测概率
        probs = []
        for smi, sp in zip(smiles_list, species_list):
            try:
                prob = predict_toxicity(model, smi, sp, species2id, scaler, device)
                probs.append(round(prob, 4))
            except:
                probs.append(np.nan)
        
        df[col_name] = probs
        print(f"✅ {col_name} 预测完成\n")

    # 保存最终结果
    df.to_excel(OUTPUT_EXCEL, index=False)
    print("🎉 全部预测完成！已自动生成 chemical ID 编号")
    print(f"📁 结果文件：{OUTPUT_EXCEL}")
    return df

# ======================================
# 运行
# ======================================
if __name__ == "__main__":
    predict_all_models()

使用设备: cpu

===== 预测: modelKE1076 =====
✅ KE1076 预测完成

===== 预测: modelKE2251 =====
✅ KE2251 预测完成

===== 预测: modelKE2294 =====
✅ KE2294 预测完成

===== 预测: modelKE531 =====
✅ KE531 预测完成

===== 预测: modelMIE2155 =====
✅ MIE2155 预测完成

===== 预测: modelMIE2293 =====
✅ MIE2293 预测完成

===== 预测: modelKE530 =====
✅ KE530 预测完成

===== 预测: modelKE1695 =====
✅ KE1695 预测完成

===== 预测: modelKE2303 =====
✅ KE2303 预测完成

===== 预测: modelKE2137 =====
✅ KE2137 预测完成

===== 预测: modelMIE1065 =====
✅ MIE1065 预测完成

===== 预测: modelKE1985 =====
✅ KE1985 预测完成

===== 预测: modelKE1047 =====
✅ KE1047 预测完成

===== 预测: modelKE2402 =====
✅ KE2402 预测完成

===== 预测: modelKE1075 =====
✅ KE1075 预测完成

===== 预测: modelKE1050 =====
✅ KE1050 预测完成

===== 预测: modelKE1972 =====
✅ KE1972 预测完成

===== 预测: modelKE1973 =====
✅ KE1973 预测完成

===== 预测: modelMIE1046 =====
✅ MIE1046 预测完成

===== 预测: modelMIE2250 =====
✅ MIE2250 预测完成

🎉 全部预测完成！已自动生成 chemical ID 编号
📁 结果文件：KE_MIE_毒性概率结果.xlsx


In [1]:
# ====================== 1. 导入依赖包 ======================
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import Data, Dataset, DataLoader
from torch_geometric.nn import SAGEConv, global_mean_pool

# ====================== 2. 固定你的图结构配置（必须完全一致） ======================
# 因果链节点
causal_chain_nodes = [
    "MIE2293", "KE1076", "KE2294", "KE2251",
    "MIE1046", "KE1047", "KE1050", "KE1972", "KE1973",
    "MIE1065", "KE1985", "KE2137", "KE2402",
    "MIE2155", "MIE2250",
    "KE530", "KE531", "KE1695", "KE2303",
    "KE1075"
]

# 因果边
causal_edges = [
    ("MIE2293", "KE1076"), ("KE1076", "KE2294"), ("KE2294", "KE2251"),
    ("MIE1046", "KE1047"), ("KE1047", "KE1050"), ("KE1050", "KE1972"), ("KE1972", "KE1973"), ("KE1973", "KE1076"),
    ("MIE1065", "KE1985"), ("KE1985", "KE1047"), ("KE1047", "KE2137"), ("KE2137", "KE2402"), ("KE2402", "KE2294"),
    ("MIE2155", "KE2251"), ("MIE2250", "KE2251"),
    ("KE530", "KE531"), ("KE531", "KE1695"), ("KE1695", "KE2303"), ("KE2303", "KE2251"),
    ("KE1075", "KE1076")
]

# 构建无向图 + 自环
undirected_edges = []
for s, d in causal_edges:
    undirected_edges.append((s, d))
    undirected_edges.append((d, s))

all_nodes = list(set(n for e in undirected_edges for n in e))
for node in all_nodes:
    undirected_edges.append((node, node))

node_id_map = {n:i for i,n in enumerate(all_nodes)}
edge_index = torch.tensor([[node_id_map[s], node_id_map[d]] for s,d in undirected_edges], dtype=torch.long).T

# 节点特征：度数 + 类型
node_degree = {node:0 for node in all_nodes}
for s,d in undirected_edges:
    if s != d:
        node_degree[s] += 1

node_type = {node:1 if node.startswith("MIE") else 0 for node in all_nodes}
degree_tensor = torch.tensor([node_degree[n] for n in all_nodes], dtype=torch.float32).unsqueeze(1)
type_tensor = torch.tensor([node_type[n] for n in all_nodes], dtype=torch.float32).unsqueeze(1)

# ====================== 3. 模型结构（必须和训练时完全一样） ======================
class GraphSAGE_Tuned(torch.nn.Module):
    def __init__(self, hidden_dim=128, dropout=0.15):
        super().__init__()
        self.conv1 = SAGEConv(3, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.conv3 = SAGEConv(hidden_dim, hidden_dim)
        
        self.bn1 = torch.nn.BatchNorm1d(hidden_dim)
        self.bn2 = torch.nn.BatchNorm1d(hidden_dim)
        self.bn3 = torch.nn.BatchNorm1d(hidden_dim)
        
        self.lin1 = torch.nn.Linear(hidden_dim, 64)
        self.lin2 = torch.nn.Linear(64, 1)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        res = x
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = x + res
        x = self.conv3(x, edge_index)
        x = self.bn3(x)
        x = F.relu(x)
        x = global_mean_pool(x, batch)
        x = self.lin1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.lin2(x)
        return x

# ====================== 4. 加载训练好的模型 ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GraphSAGE_Tuned(hidden_dim=128).to(device)
model.load_state_dict(torch.load("gcntry10.pth", map_location=device))
model.eval()
print("✅ 模型加载成功！")

# ====================== 5. 读取待预测化合物 ======================
df_pred = pd.read_excel("KE_MIE_毒性概率结果.xlsx")

# 自动匹配特征列
feature_cols = [col for col in causal_chain_nodes if col in df_pred.columns]
compound_id_col = "chemical ID" if "chemical ID" in df_pred.columns else "compound_id"

# 空值填充
df_pred[feature_cols] = df_pred[feature_cols].fillna(0)

# ====================== 6. 构建预测数据集 ======================
class PredictDataset(Dataset):
    def __init__(self, df, feature_cols, node_id_map):
        super().__init__()
        self.df = df
        self.feature_cols = feature_cols
        self.node_id_map = node_id_map
        self.num_nodes = len(node_id_map)
    
    def len(self): return len(self.df)
    def get(self, idx):
        row = self.df.iloc[idx]
        x_activate = torch.zeros((self.num_nodes,1), dtype=torch.float32)
        for col in self.feature_cols:
            x_activate[self.node_id_map[col],0] = row[col]
        x = torch.cat([x_activate, degree_tensor, type_tensor], dim=1)
        return Data(x=x, edge_index=edge_index, compound_id=row[compound_id_col])

pred_dataset = PredictDataset(df_pred, feature_cols, node_id_map)
pred_loader = DataLoader(pred_dataset, batch_size=16, shuffle=False)

# ====================== 7. 开始预测 ======================
print("🔍 正在预测...")
all_ids = []
all_probs = []

with torch.no_grad():
    for batch in pred_loader:
        batch = batch.to(device)
        logits = model(batch.x, batch.edge_index, batch.batch)
        probs = torch.sigmoid(logits).cpu().numpy().flatten()
        all_probs.extend(probs)
        all_ids.extend(batch.compound_id)

# ====================== 8. 导出结果 ======================
result_df = pd.DataFrame({
    "化合物ID": all_ids,
    "AO1065预测概率": np.round(all_probs, 4),
    "AO1065预测类别": (np.array(all_probs) >= 0.5).astype(int)
})

result_df.to_excel("AO1065结果.xlsx", index=False)
print("✅ 预测完成！文件已保存为：AO1065结果.xlsx")

D:\anaconda\envs\jupyter-clean\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\zwj19\AppData\Local\Temp\ipykernel_39476\2045899308.py:91: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend 

✅ 模型加载成功！
🔍 正在预测...
✅ 预测完成！文件已保存为：AO1065结果.xlsx


C:\Users\zwj19\AppData\Local\Temp\ipykernel_39476\2045899308.py:124: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  pred_loader = DataLoader(pred_dataset, batch_size=16, shuffle=False)
